# Market-Signal Adjusted Delta Hedging Demo

This notebook demonstrates the first implemented project workflow:

1. Generate a synthetic equity market with latent volatility regimes.
2. Construct point-in-time market signals, including option-implied volatility and volume shocks.
3. Fit a training-sample risk-state rule.
4. Convert risk states into an adjusted Black-Scholes hedge volatility path.
5. Backtest FV-BS, rolling-volatility BS, and MSA-Delta hedges on identical option episodes.

The synthetic data is only for workflow validation. Replace it with real underlying, option-chain, and market-signal data for the empirical study.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from option_hedging.backtesting import generate_option_episodes, run_strategy_comparison
from option_hedging.evaluation import summarize_hedging_performance
from option_hedging.models import RollingGaussianVolatility
from option_hedging.strategies import MarketSignalRiskModel, NORMAL, ELEVATED, STRESS


## Synthetic Market

The generated market has calm, elevated, and stress volatility regimes. Option-implied volatility is noisy but forward-looking relative to realized volatility, and volume rises during higher-risk regimes. This gives the MSA-Delta rule realistic signal structure without requiring external data for the demo.


In [ ]:
def make_synthetic_market(periods: int = 900, seed: int = 17) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2021-01-04", periods=periods)

    latent_state = np.zeros(periods, dtype=int)
    latent_state[180:250] = 1
    latent_state[430:530] = 2
    latent_state[690:760] = 1
    latent_state[760:820] = 2

    daily_vol = np.select(
        [latent_state == 0, latent_state == 1, latent_state == 2],
        [0.009, 0.017, 0.030],
    )
    drift = 0.00020 - 0.00015 * latent_state
    log_returns = drift + daily_vol * rng.standard_normal(periods)
    close = 100.0 * np.exp(np.cumsum(log_returns))

    annualized_latent_vol = daily_vol * np.sqrt(252)
    implied_vol = np.clip(
        annualized_latent_vol * (1.10 + 0.10 * latent_state)
        + 0.025 * rng.standard_normal(periods),
        0.05,
        1.50,
    )
    volume = 2_000_000 * (1.0 + 0.65 * latent_state) * np.exp(
        0.20 * rng.standard_normal(periods)
    )

    market_log_returns = 0.00012 - 0.00010 * latent_state + 0.70 * daily_vol * rng.standard_normal(periods)
    market_index = 100.0 * np.exp(np.cumsum(market_log_returns))

    return pd.DataFrame(
        {
            "date": dates,
            "close": close,
            "volume": volume,
            "implied_vol": implied_vol,
            "market_index": market_index,
            "latent_state": latent_state,
        }
    )

market = make_synthetic_market()
market.head()


## Point-in-Time Signals

Signals at date $t$ use only observations available at or before $t$:

- implied-volatility rank over the trailing 126 trading days;
- realized-volatility acceleration, measured as 21-day realized volatility divided by 63-day realized volatility;
- volume shock relative to the trailing 21-day average;
- market drawdown over the trailing 63-day window.


In [ ]:
panel = market.set_index("date")
log_returns = np.log(panel["close"] / panel["close"].shift(1)).dropna()
market_returns = np.log(panel["market_index"] / panel["market_index"].shift(1))

rv21 = log_returns.rolling(21, min_periods=21).std() * np.sqrt(252)
rv63 = log_returns.rolling(63, min_periods=63).std() * np.sqrt(252)
base_volatility = RollingGaussianVolatility(window=21).transform(log_returns).clip(lower=0.05)

iv_window_min = panel["implied_vol"].rolling(126, min_periods=63).min()
iv_window_max = panel["implied_vol"].rolling(126, min_periods=63).max()
iv_rank = ((panel["implied_vol"] - iv_window_min) / (iv_window_max - iv_window_min)).clip(0.0, 1.0)

volume_shock = np.log(panel["volume"] / panel["volume"].rolling(21, min_periods=21).mean())
market_drawdown_63 = (panel["market_index"] / panel["market_index"].rolling(63, min_periods=63).max() - 1.0).abs()

features = pd.DataFrame(
    {
        "iv_rank": iv_rank,
        "vol_ratio": rv21 / rv63,
        "volume_shock": volume_shock,
        "market_drawdown_63": market_drawdown_63,
    }
).replace([np.inf, -np.inf], np.nan).dropna()

base_volatility = base_volatility.reindex(features.index).dropna()
features = features.reindex(base_volatility.index).dropna()
base_volatility = base_volatility.reindex(features.index)

features.tail()


## Fit the MSA-Delta Risk Rule

The rule standardizes features on the training sample, computes a weighted risk score, and estimates elevated/stress thresholds from training quantiles only. The resulting thresholds are then held fixed on the test period.


In [ ]:
split_position = int(0.65 * len(features))
train_features = features.iloc[:split_position]
test_features = features.iloc[split_position:]

risk_model = MarketSignalRiskModel(
    weights={
        "iv_rank": 0.45,
        "vol_ratio": 0.25,
        "volume_shock": 0.15,
        "market_drawdown_63": 0.15,
    },
    elevated_quantile=0.70,
    stress_quantile=0.90,
    volatility_multipliers={NORMAL: 1.00, ELEVATED: 1.20, STRESS: 1.55},
).fit(train_features)

risk_diagnostics = risk_model.diagnostic_frame(test_features, base_volatility)
risk_diagnostics["risk_state"].value_counts().rename("test_state_count")


In [ ]:
risk_diagnostics.tail()


## Backtest FV-BS versus MSA-Delta

Every strategy receives the same initial option premium within each episode. That keeps the experiment focused on hedge performance rather than differences in option sale price.


In [ ]:
test_prices = panel["close"].reindex(test_features.index).dropna()
episodes = generate_option_episodes(
    test_prices,
    maturity_days=20,
    start_step=20,
    moneyness=1.0,
)

strategies = {
    "FV-BS": lambda episode: float(base_volatility.loc[episode.start_date]),
    "Rolling-BS": lambda episode: base_volatility.reindex(episode.prices.index[:-1]),
    "MSA-Delta": lambda episode: risk_diagnostics["msa_delta_volatility"].reindex(
        episode.prices.index[:-1]
    ),
}

episode_results, hedge_paths = run_strategy_comparison(
    episodes,
    strategies=strategies,
    pricing_strategy="FV-BS",
    rate=0.03,
    transaction_cost_rate=0.0005,
)

hedging_metrics = summarize_hedging_performance(episode_results)
hedging_metrics.round(6)


In [ ]:
episode_results.head()


## Save Demo Outputs

The output tables are useful for quick inspection and can be replaced by real-data results later.


In [ ]:
output_dir = PROJECT_ROOT / "results" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

episode_results.to_csv(output_dir / "msa_delta_demo_episode_results.csv", index=False)
hedge_paths.to_csv(output_dir / "msa_delta_demo_hedge_paths.csv", index=False)
hedging_metrics.to_csv(output_dir / "msa_delta_demo_hedging_metrics.csv")
risk_diagnostics.to_csv(output_dir / "msa_delta_demo_risk_diagnostics.csv")

sorted(path.name for path in output_dir.glob("msa_delta_demo_*.csv"))
